In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import os
import time
import random
from urllib.parse import urljoin

BASE_URL = "https://www.recetasgratis.net"
OUTPUT_JSON = "recetas.json"
MAX_RECETAS = 10000
REQUEST_DELAY = (0.6, 1.0)

CATEGORIAS = [
    "Recetas-de-Pescado-listado_receta-12_1.html",
    "Recetas-de-Postres-listado_receta-17_1.html",
    "Recetas-de-Sopa-listado_receta-6_1.html",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
}

# ---------------- Utilidades ----------------

def safe_get(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
        return r.text
    except:
        return None

def load_existing_data():
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_data(data):
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

# ---------------- Scraper ----------------

def extract_links(html):
    soup = BeautifulSoup(html, "html.parser")
    anchors = soup.select("a.titulo.titulo--resultado")
    return [urljoin(BASE_URL, a["href"]) for a in anchors if a.get("href")]

def parse_recipe(html, url):
    soup = BeautifulSoup(html, "html.parser")

    # Nombre
    nombre = soup.select_one("h1.titulo")
    nombre = nombre.get_text(" ", strip=True) if nombre else ""

    # Comensales, Tiempo, Dificultad
    comensales = soup.select_one(".property.comensales")
    comensales = comensales.get_text(" ", strip=True) if comensales else ""

    tiempo = soup.select_one(".property.duracion")
    tiempo = tiempo.get_text(" ", strip=True) if tiempo else ""

    dificultad = soup.select_one(".property.dificultad")
    dificultad = dificultad.get_text(" ", strip=True) if dificultad else ""

    # Ingredientes (una lista con cantidades y nombres juntos)
    ingredientes = [li.get_text(" ", strip=True) for li in soup.select(".ingredientes li")]

    # Instrucciones: cada div.apartado es un paso
    pasos = []
    for div in soup.select("div.apartado"):
        textos = [p.get_text(" ", strip=True) for p in div.find_all("p") if p.get_text(strip=True)]
        if textos:
            paso_unido = " ".join(textos)
            paso_unido = " ".join(paso_unido.split()) 
            pasos.append(paso_unido)

    return {
        "Nombre": nombre,
        "Comensales": comensales,
        "Dificultad": dificultad,
        "Tiempo": tiempo,
        "Ingredientes": ingredientes,
        "Instrucciones": pasos
    }

# ---------------- Flujo principal ----------------

def scrape_recipes():
    data = load_existing_data()
    processed_urls = {r.get("url") for r in data if "url" in r}
    total = len(data)

    for categoria in CATEGORIAS:
        if total >= MAX_RECETAS:
            break
        page = 1
        while total < MAX_RECETAS:
            url_listado = f"{BASE_URL}/{categoria.replace('_1.html', f'_{page}.html')}"
            html_listado = safe_get(url_listado)
            if not html_listado:
                break

            links = extract_links(html_listado)
            if not links:
                break

            for link in links:
                if total >= MAX_RECETAS:
                    break
                if link in processed_urls:
                    continue

                html_receta = safe_get(link)
                if not html_receta:
                    continue

                receta = parse_recipe(html_receta, link)
                receta["url"] = link 
                data.append(receta)
                processed_urls.add(link)
                total += 1
                print(f"[{total}] {receta['Nombre']}")
                save_data(data) 
                time.sleep(random.uniform(*REQUEST_DELAY))

            page += 1

    for r in data:
        r.pop("url", None)
    save_data(data)
    print(f"Total recetas guardadas: {len(data)}")

if __name__ == "__main__":
    scrape_recipes()


Total recetas guardadas: 8272
